# *05 SQL and Python Analysis Requirements*

### Minimum Python Notebook Tasks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
events = pd.read_csv(
    "Events.csv.csv"
)

orders = pd.read_csv(
    "Orders.csv.csv"
)

In [ ]:
events.head()

In [ ]:
orders.head()

In [ ]:
print("Events Shape:", events.shape)
print("Orders Shape:", orders.shape)

Q1. Clean and join event and order datasets.

In [ ]:
print("Events Missing Values:")
print(events.isnull().sum())

print("\nOrders Missing Values:")
print(orders.isnull().sum())

- Remove Exact Duplicates

In [ ]:
events = events.drop_duplicates()
orders = orders.drop_duplicates()

In [ ]:
print("Event Duplicates:", events.duplicated().sum())
print("Order Duplicates:", orders.duplicated().sum())

- Clean Text Columns

In [ ]:
for col in events.select_dtypes(include='object').columns:
    events[col] = events[col].str.strip()

for col in orders.select_dtypes(include='object').columns:
    orders[col] = orders[col].str.strip()

- Convert Date Columns

In [ ]:
events['event_time'] = pd.to_datetime(events['event_time'], format="%d-%m-%Y")

orders['order_date'] = pd.to_datetime(orders['order_date'], format="%d-%m-%Y")

In [ ]:
events.info()
orders.info()

- Join Events and Orders

In [ ]:
order_info = orders[[
    'order_id',
    'customer_id',
    'net_revenue',
    'return_flag'
]].drop_duplicates(subset='order_id')

In [ ]:
combined_data = events.merge(
    order_info,
    on='order_id',
    how='left',
    suffixes=('_event', '_order')
)

In [ ]:
combined_data.head()

In [ ]:
print("Combined Dataset Shape:", combined_data.shape)

Q2. Validate Funnel-Event Sequence Assumptions

- Create Funnel Order

In [ ]:
funnel_order = {
    'view': 1,
    'add_to_cart': 2,
    'checkout': 3,
    'purchase': 4
}

In [ ]:
events['funnel_stage'] = events['event_type'].map(funnel_order)

- Check Event Sequence by Session

In [ ]:
events = events.sort_values(
    ['session_id', 'event_time']
)

In [ ]:
sequence_check = events.groupby('session_id')['funnel_stage'].apply(
    lambda x: x.is_monotonic_increasing
)

print(sequence_check.value_counts())

In [ ]:
valid_sequence_percent = sequence_check.mean() * 100

print(
    "Valid Funnel Sequence Percentage:",
    round(valid_sequence_percent, 2),
    "%"
)

Q3. Create Funnel Visualization

- Calculate Funnel Counts

In [ ]:
funnel_counts = (
    events['event_type']
    .value_counts()
    .reindex([
        'view',
        'add_to_cart',
        'checkout',
        'purchase'
    ])
)

print(funnel_counts)

- Create Funnel Chart

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    funnel_counts.index,
    funnel_counts.values
)

plt.title("E-Commerce Conversion Funnel")
plt.xlabel("Funnel Stage")
plt.ylabel("Number of Events")

plt.show()

Q4. Plot Channel Conversion

- Calculate Total Sessions by Channel

In [ ]:
channel_sessions = (
    events.groupby('traffic_source')['session_id']
    .nunique()
)

- Calculate Purchase Sessions

In [ ]:
channel_purchases = (
    events[
        events['event_type'] == 'purchase'
    ]
    .groupby('traffic_source')['session_id']
    .nunique()
)

- Calculate Conversion


In [ ]:
channel_conversion = pd.DataFrame({
    'total_sessions': channel_sessions,
    'purchase_sessions': channel_purchases
}).fillna(0)

channel_conversion['conversion_rate'] = (
    channel_conversion['purchase_sessions']
    / channel_conversion['total_sessions']
    * 100
)

print(channel_conversion)

- Plot Channel Conversion

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    channel_conversion.index,
    channel_conversion['conversion_rate']
)

plt.title("Conversion Rate by Traffic Source")
plt.xlabel("Traffic Source")
plt.ylabel("Conversion Rate (%)")

plt.show()

Q5. Plot Device Conversion

- Total Sessions by Device

In [ ]:
device_sessions = (
    events.groupby('device_type')['session_id']
    .nunique()
)

- Purchase Sessions by Device

In [ ]:
device_purchases = (
    events[
        events['event_type'] == 'purchase'
    ]
    .groupby('device_type')['session_id']
    .nunique()
)

- Calculate Conversion Rate

In [ ]:
device_conversion = pd.DataFrame({
    'total_sessions': device_sessions,
    'purchase_sessions': device_purchases
}).fillna(0)

device_conversion['conversion_rate'] = (
    device_conversion['purchase_sessions']
    / device_conversion['total_sessions']
    * 100
)

print(device_conversion)

- Create Chart

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    device_conversion.index,
    device_conversion['conversion_rate']
)

plt.title("Conversion Rate by Device")
plt.xlabel("Device Type")
plt.ylabel("Conversion Rate (%)")

plt.show()

Q6. Analyze Repeat Customer Behavior

- Count Orders per Customer

In [ ]:
customer_orders = (
    orders.groupby('customer_id')['order_id']
    .nunique()
    .reset_index()
)

customer_orders.columns = [
    'customer_id',
    'total_orders'
]

- Create Customer Type

In [ ]:
customer_orders['customer_type'] = np.where(
    customer_orders['total_orders'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)

In [ ]:
customer_orders.head()

- Count Customer Types

In [ ]:
repeat_summary = (
    customer_orders['customer_type']
    .value_counts()
)

print(repeat_summary)

- Calculate Repeat Customer Rate

In [ ]:
repeat_customer_rate = (
    (customer_orders['customer_type'] == 'Repeat Customer').mean()
    * 100
)

print(
    "Repeat Customer Rate:",
    round(repeat_customer_rate, 2),
    "%"
)

- Create Visualization

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    repeat_summary.index,
    repeat_summary.values
)

plt.title("Repeat vs One-Time Customers")
plt.xlabel("Customer Type")
plt.ylabel("Number of Customers")

plt.show()

Q7. Create Category Performance Scatterplot

- Category Views

In [ ]:
category_views = (
    events[
        events['event_type'] == 'view'
    ]
    .groupby('category')
    .size()
)

- Category Purchases

In [ ]:
category_purchases = (
    events[
        events['event_type'] == 'purchase'
    ]
    .groupby('category')
    .size()
)

- Category Revenue

In [ ]:
category_revenue = (
    orders.groupby('category')['net_revenue']
    .sum()
)

- Combine Results

In [ ]:
category_performance = pd.DataFrame({
    'views': category_views,
    'purchases': category_purchases,
    'revenue': category_revenue
}).fillna(0)

print(category_performance)

- Create Scatterplot

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    category_performance['views'],
    category_performance['purchases'],
    s=category_performance['revenue'] / 100
)

for category, row in category_performance.iterrows():
    plt.annotate(
        category,
        (row['views'], row['purchases'])
    )

plt.title("Category Performance: Views vs Purchases")
plt.xlabel("Product Views")
plt.ylabel("Purchases")

plt.show()

Q8. Investigate Return-Rate Outliers

- Calculate Product Return Rate

In [ ]:
return_analysis = (
    orders.groupby('product_name')
    .agg(
        total_order_lines=('order_id', 'count'),
        returned_items=(
            'return_flag',
            lambda x: (x == 'Yes').sum()
        ),
        total_revenue=('net_revenue', 'sum')
    )
)

- Calculate Return Rate

In [ ]:
return_analysis['return_rate'] = (
    return_analysis['returned_items']
    / return_analysis['total_order_lines']
    * 100
)

print(
    return_analysis.sort_values(
        'return_rate',
        ascending=False
    ).head(10)
)

- Detect Outliers Using IQR Method

In [ ]:
Q1 = return_analysis['return_rate'].quantile(0.25)

Q3 = return_analysis['return_rate'].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

outliers = return_analysis[
    return_analysis['return_rate'] > upper_limit
]

print("Return Rate Outliers:")
print(outliers)

- Plot Return Rates

In [ ]:
top_returns = (
    return_analysis
    .sort_values('return_rate', ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 6))

plt.bar(
    top_returns.index,
    top_returns['return_rate']
)

plt.title("Top Products by Return Rate")
plt.xlabel("Product Name")
plt.ylabel("Return Rate (%)")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

#  ***Complete E-Commerce Analytics Dashboard***





In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install openpyxl plotly ipywidgets -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output
import ipywidgets as widgets

In [ ]:
csv_files = [
    'Events.csv.csv',
    'Orders.csv.csv'
]

print("Available CSV files:")
for file in csv_files:
    try:
        df = pd.read_csv(file)
        print(f"- {file} (Loaded successfully with {len(df)} rows)")
    except FileNotFoundError:
        print(f"- {file} (Not found)")
    except Exception as e:
        print(f"- {file} (Error loading: {e})")

In [ ]:
events = pd.read_csv(
    "Events.csv.csv"
)

orders = pd.read_csv(
    "Orders.csv.csv"
)

print("Events Shape:", events.shape)
print("Orders Shape:", orders.shape)

In [ ]:
print("EVENTS COLUMNS:")
print(events.columns.tolist())

print("\nORDERS COLUMNS:")
print(orders.columns.tolist())

In [ ]:
events.columns = (
    events.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Cleaned Event Columns:")
print(events.columns.tolist())

print("\nCleaned Order Columns:")
print(orders.columns.tolist())

In [ ]:
# Remove completely duplicate rows
events = events.drop_duplicates()
orders = orders.drop_duplicates()

# Convert dates
events["event_time"] = pd.to_datetime(
    events["event_time"],
    errors="coerce"
)

orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

# Convert revenue to numeric
orders["net_revenue"] = pd.to_numeric(
    orders["net_revenue"],
    errors="coerce"
)

# Remove missing important values
events = events.dropna(
    subset=["session_id", "event_type"]
)

orders = orders.dropna(
    subset=["order_id", "customer_id"]
)

print("Data Cleaning Completed!")

In [ ]:
events["event_month"] = (
    events["event_time"]
    .dt.to_period("M")
    .astype(str)
)

orders["order_month"] = (
    orders["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
def create_dashboard(
    selected_source="All",
    selected_device="All",
    selected_category="All"
):

    # ------------------------------------------
    # CREATE COPIES
    # ------------------------------------------

    filtered_events = events.copy()
    filtered_orders = orders.copy()


    # ------------------------------------------
    # APPLY FILTERS
    # ------------------------------------------

    if selected_source != "All":
        filtered_events = filtered_events[
            filtered_events["traffic_source"] == selected_source
        ]

    if selected_device != "All":
        filtered_events = filtered_events[
            filtered_events["device_type"] == selected_device
        ]

    if selected_category != "All":
        filtered_events = filtered_events[
            filtered_events["category"] == selected_category
        ]


    # ------------------------------------------
    # FILTER ORDERS USING ORDER ID
    # ------------------------------------------

    if "order_id" in filtered_events.columns:

        order_ids = (
            filtered_events["order_id"]
            .dropna()
            .unique()
        )

        if len(order_ids) > 0:
            filtered_orders = filtered_orders[
                filtered_orders["order_id"].isin(order_ids)
            ]


    # ==========================================
    # KPI CALCULATIONS
    # ==========================================

    total_sessions = (
        filtered_events["session_id"]
        .nunique()
    )


    product_views = len(
        filtered_events[
            filtered_events["event_type"] == "view"
        ]
    )


    add_to_cart_events = len(
        filtered_events[
            filtered_events["event_type"] == "add_to_cart"
        ]
    )


    checkout_events = len(
        filtered_events[
            filtered_events["event_type"] == "checkout"
        ]
    )


    purchase_events = len(
        filtered_events[
            filtered_events["event_type"] == "purchase"
        ]
    )


    # Add-to-Cart Rate

    add_to_cart_rate = (
        add_to_cart_events / product_views * 100
        if product_views > 0
        else 0
    )


    # Checkout Rate

    checkout_rate = (
        checkout_events / add_to_cart_events * 100
        if add_to_cart_events > 0
        else 0
    )


    # Purchase Sessions

    purchase_sessions = (
        filtered_events[
            filtered_events["event_type"] == "purchase"
        ]["session_id"]
        .nunique()
    )


    # Purchase Conversion

    purchase_conversion = (
        purchase_sessions / total_sessions * 100
        if total_sessions > 0
        else 0
    )


    # Cart Sessions

    cart_sessions = (
        filtered_events[
            filtered_events["event_type"] == "add_to_cart"
        ]["session_id"]
        .nunique()
    )


    # Cart Abandonment

    cart_abandonment = (
        (cart_sessions - purchase_sessions)
        / cart_sessions * 100
        if cart_sessions > 0
        else 0
    )


    # Total Orders

    total_orders = (
        filtered_orders["order_id"]
        .nunique()
    )


    # Total Revenue

    total_revenue = (
        filtered_orders["net_revenue"]
        .sum()
    )


    # Average Order Value

    average_order_value = (
        total_revenue / total_orders
        if total_orders > 0
        else 0
    )


    # Repeat Customer Rate

    customer_order_count = (
        filtered_orders
        .groupby("customer_id")["order_id"]
        .nunique()
    )


    repeat_customers = (
        customer_order_count > 1
    ).sum()


    total_customers = (
        customer_order_count.count()
    )


    repeat_customer_rate = (
        repeat_customers / total_customers * 100
        if total_customers > 0
        else 0
    )


    # Return Rate

    if "return_flag" in filtered_orders.columns:

        returned_orders = (
            filtered_orders[
                filtered_orders["return_flag"]
                .astype(str)
                .str.lower()
                .isin(["yes", "returned", "return"])
            ]["order_id"]
            .nunique()
        )

        return_rate = (
            returned_orders / total_orders * 100
            if total_orders > 0
            else 0
        )

    else:
        return_rate = 0


    # ==========================================
    # DASHBOARD TITLE
    # ==========================================

    print("=" * 90)
    print("             🛒 E-COMMERCE PERFORMANCE ANALYTICS DASHBOARD")
    print("=" * 90)


    # ==========================================
    # KPI CARDS
    # ==========================================

    kpi_data = pd.DataFrame({
        "KPI": [
            "Total Sessions",
            "Product Views",
            "Add-to-Cart Rate %",
            "Checkout Rate %",
            "Purchase Conversion %",
            "Cart Abandonment %",
            "Total Orders",
            "Total Revenue",
            "Average Order Value",
            "Repeat Customer Rate %",
            "Return Rate %"
        ],

        "Value": [
            f"{total_sessions:,}",
            f"{product_views:,}",
            f"{add_to_cart_rate:.2f}%",
            f"{checkout_rate:.2f}%",
            f"{purchase_conversion:.2f}%",
            f"{cart_abandonment:.2f}%",
            f"{total_orders:,}",
            f"₹{total_revenue:,.2f}",
            f"₹{average_order_value:,.2f}",
            f"{repeat_customer_rate:.2f}%",
            f"{return_rate:.2f}%"
        ]
    })

    display(kpi_data)


    # ==========================================
    # FUNNEL CHART
    # ==========================================

    funnel_data = pd.DataFrame({
        "Stage": [
            "Product Views",
            "Add to Cart",
            "Checkout",
            "Purchase"
        ],

        "Count": [
            product_views,
            add_to_cart_events,
            checkout_events,
            purchase_events
        ]
    })

    funnel_fig = px.funnel(
        funnel_data,
        y="Stage",
        x="Count",
        title="E-Commerce Conversion Funnel"
    )

    funnel_fig.show()


    # ==========================================
    # TRAFFIC SOURCE CONVERSION
    # ==========================================

    channel_sessions = (
        filtered_events
        .groupby("traffic_source")["session_id"]
        .nunique()
    )

    channel_purchase_sessions = (
        filtered_events[
            filtered_events["event_type"] == "purchase"
        ]
        .groupby("traffic_source")["session_id"]
        .nunique()
    )

    channel_conversion = pd.DataFrame({
        "Total Sessions": channel_sessions,
        "Purchase Sessions": channel_purchase_sessions
    }).fillna(0)

    channel_conversion["Conversion Rate (%)"] = (
        channel_conversion["Purchase Sessions"]
        / channel_conversion["Total Sessions"]
        * 100
    )

    channel_conversion = (
        channel_conversion
        .reset_index()
    )

    channel_fig = px.bar(
        channel_conversion,
        x="traffic_source",
        y="Conversion Rate (%)",
        text_auto=".2f",
        title="Conversion Rate by Traffic Source"
    )

    channel_fig.show()


    # ==========================================
    # DEVICE CONVERSION
    # ==========================================

    device_sessions = (
        filtered_events
        .groupby("device_type")["session_id"]
        .nunique()
    )

    device_purchase_sessions = (
        filtered_events[
            filtered_events["event_type"] == "purchase"
        ]
        .groupby("device_type")["session_id"]
        .nunique()
    )

    device_conversion = pd.DataFrame({
        "Total Sessions": device_sessions,
        "Purchase Sessions": device_purchase_sessions
    }).fillna(0)

    device_conversion["Conversion Rate (%)"] = (
        device_conversion["Purchase Sessions"]
        / device_conversion["Total Sessions"]
        * 100
    )

    device_conversion = (
        device_conversion
        .reset_index()
    )

    device_fig = px.bar(
        device_conversion,
        x="device_type",
        y="Conversion Rate (%)",
        text_auto=".2f",
        title="Conversion Rate by Device"
    )

    device_fig.show()


    # ==========================================
    # MONTHLY REVENUE
    # ==========================================

    monthly_revenue = (
        filtered_orders
        .groupby("order_month")["net_revenue"]
        .sum()
        .reset_index()
    )

    revenue_fig = px.line(
        monthly_revenue,
        x="order_month",
        y="net_revenue",
        markers=True,
        title="Monthly Revenue Trend"
    )

    revenue_fig.show()


    # ==========================================
    # CATEGORY REVENUE
    # ==========================================

    if "category" in filtered_orders.columns:

        category_revenue = (
            filtered_orders
            .groupby("category")["net_revenue"]
            .sum()
            .reset_index()
        )

        category_fig = px.pie(
            category_revenue,
            names="category",
            values="net_revenue",
            title="Revenue by Category"
        )

        category_fig.show()


    # ==========================================
    # TOP 10 PRODUCTS
    # ==========================================

    if "product_name" in filtered_orders.columns:

        top_products = (
            filtered_orders
            .groupby("product_name")["net_revenue"]
            .sum()
            .sort_values(ascending=False)
            .head(10)
            .reset_index()
        )

        product_fig = px.bar(
            top_products,
            x="net_revenue",
            y="product_name",
            orientation="h",
            title="Top 10 Products by Revenue"
        )

        product_fig.show()


    # ==========================================
    # REPEAT CUSTOMER ANALYSIS
    # ==========================================

    customer_summary = (
        filtered_orders
        .groupby("customer_id")["order_id"]
        .nunique()
        .reset_index()
    )

    customer_summary.columns = [
        "customer_id",
        "total_orders"
    ]

    customer_summary["customer_type"] = np.where(
        customer_summary["total_orders"] > 1,
        "Repeat Customer",
        "One-Time Customer"
    )

    customer_counts = (
        customer_summary["customer_type"]
        .value_counts()
        .reset_index()
    )

    customer_counts.columns = [
        "Customer Type",
        "Customers"
    ]

    customer_fig = px.pie(
        customer_counts,
        names="Customer Type",
        values="Customers",
        title="Repeat vs One-Time Customers"
    )

    customer_fig.show()


    # ==========================================
    # RETURN RATE BY PRODUCT
    # ==========================================

    if (
        "return_flag" in filtered_orders.columns
        and "product_name" in filtered_orders.columns
    ):

        return_analysis = (
            filtered_orders
            .groupby("product_name")
            .agg(
                Total_Orders=("order_id", "nunique"),

                Returned_Orders=(
                    "return_flag",
                    lambda x: (
                        x.astype(str)
                        .str.lower()
                        .isin(["yes", "returned", "return"])
                        .sum()
                    )
                )
            )
            .reset_index()
        )

        return_analysis["Return Rate (%)"] = (
            return_analysis["Returned_Orders"]
            / return_analysis["Total_Orders"]
            * 100
        )

        top_returns = (
            return_analysis
            .sort_values(
                "Return Rate (%)",
                ascending=False
            )
            .head(10)
        )

        return_fig = px.bar(
            top_returns,
            x="Return Rate (%)",
            y="product_name",
            orientation="h",
            title="Top Products by Return Rate"
        )

        return_fig.show()


    # ==========================================
    # INSIGHTS
    # ==========================================

    print("\n" + "=" * 90)
    print("📌 DASHBOARD SUMMARY")
    print("=" * 90)

    print(f"Total Sessions: {total_sessions:,}")
    print(f"Total Orders: {total_orders:,}")
    print(f"Total Revenue: ₹{total_revenue:,.2f}")
    print(f"Purchase Conversion: {purchase_conversion:.2f}%")
    print(f"Cart Abandonment: {cart_abandonment:.2f}%")
    print(f"Repeat Customer Rate: {repeat_customer_rate:.2f}%")
    print(f"Return Rate: {return_rate:.2f}%")

In [ ]:
traffic_sources = ["All"] + sorted(
    events["traffic_source"]
    .dropna()
    .unique()
    .tolist()
)

devices = ["All"] + sorted(
    events["device_type"]
    .dropna()
    .unique()
    .tolist()
)

categories = ["All"] + sorted(
    events["category"]
    .dropna()
    .unique()
    .tolist()
)


source_dropdown = widgets.Dropdown(
    options=traffic_sources,
    value="All",
    description="Traffic Source:"
)

device_dropdown = widgets.Dropdown(
    options=devices,
    value="All",
    description="Device:"
)

category_dropdown = widgets.Dropdown(
    options=categories,
    value="All",
    description="Category:"
)


ui = widgets.VBox([
    source_dropdown,
    device_dropdown,
    category_dropdown
])


out = widgets.interactive_output(
    create_dashboard,
    {
        "selected_source": source_dropdown,
        "selected_device": device_dropdown,
        "selected_category": category_dropdown
    }
)


display(ui, out)